# Test MedBook API

This notebook demonstrates how to interact with the MedBook API using both approaches:
1. **Stateless REST** (`POST /chat`)
2. **Stateful WebSocket** (`ws://.../ws/chat`)

In [2]:
import requests
import asyncio
import websockets
import json
from pprint import pprint

## 1. Stateless REST (`POST /chat`)
In this approach, you must pass the `messages` array back to the server in every request to maintain the conversation history.

In [3]:
def test_rest_api():
    url = "http://localhost:8000/chat"
    
    # Turn 1: Fresh conversation
    print("\n--- Turn 1 ---")
    payload1 = {"user_message": "I have a headache and need to see a doctor"}
    response1 = requests.post(url, json=payload1).json()
    print("Bot:", response1.get("reply"))
    
    # Save messages for the next turn
    messages = response1.get("messages", [])
    
    # Turn 2: Provide phone number
    print("\n--- Turn 2 ---")
    payload2 = {
        "user_message": "8952039590",
        "messages": messages
    }
    response2 = requests.post(url, json=payload2).json()
    print("Bot:", response2.get("reply"))
    
    # Turn 3: Provide name (if asked)
    print("\n--- Turn 3 ---")
    messages = response2.get("messages", [])
    payload3 = {
        "user_message": "Harsh",
        "messages": messages
    }
    response3 = requests.post(url, json=payload3).json()
    print("Bot:", response3.get("reply"))

    # Add more turns as needed...
    
# Uncomment to run (Make sure API is running!)
test_rest_api()


--- Turn 1 ---
Bot: I can help with that! First, could you please provide me with your phone number?

--- Turn 2 ---
Bot: I recommend seeing a General Physician for your headache. 

I found a doctor for you:
- **Dr. Amit Sharma**
  - **Speciality:** General Physician
  - **Qualifications:** MBBS, DNB (Internal Medicine)
  - **Experience:** 22+ years
  - **Languages:** Hindi, English

Would you like to book an appointment with Dr. Amit Sharma? If so, please let me know what date works for you and your preferred time of day (morning, afternoon, or evening).

--- Turn 3 ---
Bot: Thank you for confirming your name, Harsh! What date would you like to book your appointment with Dr. Amit Sharma?


## 2. Stateful WebSocket (`ws://localhost:8000/ws/chat`)
In this approach, the server maintains the state (`messages` array) in memory for the duration of the connection.

In [5]:
async def test_websocket_api():
    uri = "ws://localhost:8000/ws/chat"
    async with websockets.connect(uri) as ws:
        # Initial connection message from server
        greeting_raw = await ws.recv()
        greeting = json.loads(greeting_raw)
        print("Server:", greeting.get("message"))
        print("Session ID:", greeting.get("session_id"))
        
        # Turn 1
        print("\n--- Turn 1 ---")
        user_msg = "I have a headache"
        print("You:", user_msg)
        await ws.send(user_msg)
        reply_raw = await ws.recv()
        reply = json.loads(reply_raw)
        print("Bot:", reply.get("message"))
        
        # Turn 2
        print("\n--- Turn 2 ---")
        user_msg = "8952039590"
        print("You:", user_msg)
        await ws.send(user_msg)
        reply_raw = await ws.recv()
        reply = json.loads(reply_raw)
        print("Bot:", reply.get("message"))
        
        # Turn 3
        print("\n--- Turn 3 ---")
        user_msg = "Harsh"
        print("You:", user_msg)
        await ws.send(user_msg)
        reply_raw = await ws.recv()
        reply = json.loads(reply_raw)
        print("Bot:", reply.get("message"))

        # Reset session example
        print("\n--- Resetting Session ---")
        await ws.send("reset")
        reset_raw = await ws.recv()
        reset_reply = json.loads(reset_raw)
        print("Server:", reset_reply.get("message"))
        
# Uncomment to run (Jupyter manages its own event loop, so we can await directly in newer Jupyter versions or use asyncio.run if needed outside)
await test_websocket_api()

Server: Connected to MedBook API.
Session ID: 286eb9d5-50c7-40d2-9a69-186406895e20

--- Turn 1 ---
You: I have a headache
Bot: I'm sorry to hear that you're experiencing a headache. To help you find the right specialist, could you please provide me with your phone number?

--- Turn 2 ---
You: 8952039590
Bot: For your headache, I recommend seeing a General Physician. 

**Doctor Available:**
- **Name:** Dr. Amit Sharma
- **Speciality:** General Physician
- **Qualifications:** MBBS, DNB (Internal Medicine)
- **Experience:** 22+ years
- **Languages:** Hindi, English

Would you like to check available appointment slots with Dr. Amit Sharma? If so, please let me know what date works for you!

--- Turn 3 ---
You: Harsh
Bot: Thank you, Harsh! What date would you like to schedule your appointment with Dr. Amit Sharma?

--- Resetting Session ---
Server: --- Conversation reset. Start fresh! ---
